# Keep/Cull RL Deney Runner

Bu notebook MDP, karşılaştırmalı RL ve Gymnasium + Stable-Baselines3 PPO/DQN deneylerini parametre değiştirerek çalıştırmak için hazırlanmıştır.

Ana kod katmanı: `Model/rl/code/`

Ana config: `config/rl_env_rules.yaml`

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yaml

cwd = Path.cwd().resolve()
if (cwd / 'config' / 'rl_env_rules.yaml').exists():
    PROJECT_ROOT = cwd
else:
    matches = list(cwd.glob('KDS/Sürüden_keep_cull')) + list(cwd.parents[0].glob('KDS/Sürüden_keep_cull')) if cwd.parents else []
    PROJECT_ROOT = matches[0].resolve() if matches else (cwd.parent if (cwd.parent / 'config' / 'rl_env_rules.yaml').exists() else cwd)

CODE_DIR = PROJECT_ROOT / 'Model' / 'rl' / 'code'
sys.path.insert(0, str(CODE_DIR))

from experiment_runner import (
    PARAMETERS_PATH,
    CONFIG_PATH,
    apply_overrides,
    load_yaml,
    save_yaml,
    run_mdp_pipeline,
    run_comparative_rl_pipeline,
    run_sb3_pipeline,
    summarize_outputs,
    load_output_tables,
)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CONFIG_PATH  =', CONFIG_PATH)
print('PARAMETERS   =', PARAMETERS_PATH)

## 1. Parametreleri Ayarla

Aşağıdaki değerleri değiştirip hücreyi çalıştır. Hücre `config/rl_env_rules.yaml` içine yazar.

In [ ]:
params = load_yaml(PARAMETERS_PATH)

# MDP
params['overrides']['mdp.discount_factor'] = 0.97
params['overrides']['mdp.convergence_tolerance'] = 0.0001
params['overrides']['mdp.max_iterations'] = 1000

# Tabular / smoke RL
params['overrides']['rl_experiments.algorithms.tabular_q_learning.episodes'] = 4000
params['overrides']['rl_experiments.algorithms.tabular_ppo_smoke.iterations'] = 140
params['overrides']['rl_experiments.algorithms.dqn_style_fqi_mlp.fitted_iterations'] = 10

# Stable-Baselines3 gerçek eğitim
params['overrides']['stable_baselines3.eval_episodes'] = 100
params['overrides']['stable_baselines3.algorithms.PPO.total_timesteps'] = 5000
params['overrides']['stable_baselines3.algorithms.DQN.total_timesteps'] = 5000

# SB3 dependency olan Python. Yoksa script missing_dependency raporu üretir.
params['python']['sb3'] = '/tmp/laktasens-rl-venv/bin/python'

save_yaml(PARAMETERS_PATH, params)
apply_overrides(CONFIG_PATH, params['overrides'])
params

## 2. MDP Pipeline Çalıştır

Reward table, value iteration policy ve scenario sensitivity üretir.

In [ ]:
run_mdp_pipeline(sys.executable)
display(summarize_outputs())

## 3. Karşılaştırmalı RL Deneyleri

Readiness audit, heuristic benchmark, tabular Q-learning, tabular PPO smoke ve DQN-style FQI karşılaştırmasını çalıştırır.

In [ ]:
run_comparative_rl_pipeline(sys.executable)
tables = load_output_tables()
display(tables.get('algorithm_comparison', pd.DataFrame()))

## 4. Gymnasium + Stable-Baselines3 PPO/DQN Eğitimi

Gereken ortam yoksa önce terminalde:

```bash
cd KDS/Sürüden_keep_cull
python3 -m venv /tmp/laktasens-rl-venv
/tmp/laktasens-rl-venv/bin/python -m pip install --upgrade pip wheel
/tmp/laktasens-rl-venv/bin/python -m pip install -r requirements-rl.txt
```

Sonra aşağıdaki hücreyi çalıştır.

In [ ]:
SB3_PYTHON = params['python']['sb3']
run_sb3_pipeline(SB3_PYTHON)
tables = load_output_tables()
display(tables.get('sb3_results', pd.DataFrame()))

## 5. Sonuçları Oku

In [ ]:
tables = load_output_tables()
for name, table in tables.items():
    print('\n###', name, table.shape)
    display(table.head(20))